# HybridFrame Demo

This notebook demonstrates the key features of HybridFrame - a high-performance DataFrame that unifies DuckDB and Pandas.

In [ ]:
from hybrid_frame import HybridFrame
import pandas as pd
import numpy as np

# Create sample data
np.random.seed(42)
n = 100000
df = pd.DataFrame({
    'id': range(n),
    'name': np.random.choice(['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'], n),
    'age': np.random.randint(18, 65, n),
    'salary': np.random.uniform(30000, 120000, n),
    'dept': np.random.choice(['Engineering', 'Marketing', 'Sales', 'HR', 'Finance'], n),
    'performance': np.random.choice(['Excellent', 'Good', 'Average', 'Poor'], n)
})

print(f"Created DataFrame with {len(df):,} rows")
df.head()

## 1. Basic Operations

In [ ]:
# Create HybridFrame from Pandas
hf = HybridFrame.from_pandas(df)
print(f"HybridFrame shape: {hf.shape}")
print(f"Engine: {hf._engine}")

In [ ]:
# Filter: stays in DuckDB mode (lazy)
hf.filter("age > 30")
print(f"After filter - Shape: {hf.shape}")
print(f"Engine: {hf._engine}")

In [ ]:
# Select: transitions to Pandas mode (zero-copy)
hf.select(['name', 'age', 'salary'])
print(f"After select - Shape: {hf.shape}")
print(f"Engine: {hf._engine}")

## 2. Chained Operations

In [ ]:
# Complex query chain
result = (
    HybridFrame.from_pandas(df)
    .filter("age > 25")
    .filter("salary > 50000")
    .sort_values('salary', ascending=False)
    .head(10)
)

print(f"Result shape: {result.shape}")
result.to_pandas()

## 3. Aggregations

In [ ]:
# GroupBy aggregation
agg_result = (
    HybridFrame.from_pandas(df)
    .groupby_agg(['dept'], {'salary': ['sum', 'mean', 'max']})
)

agg_result.to_pandas()

## 4. Joins

In [ ]:
# Create second DataFrame for joining
dept_info = pd.DataFrame({
    'dept': ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance'],
    'budget': [1000000, 500000, 750000, 300000, 400000],
    'headcount': [50, 30, 40, 15, 20]
})

# Join
joined = (
    HybridFrame.from_pandas(df)
    .join(HybridFrame.from_pandas(dept_info), on='dept')
    .select(['name', 'dept', 'salary', 'budget'])
    .head(10)
)

joined.to_pandas()

## 5. Data Cleaning

In [ ]:
# Create DataFrame with missing values
df_dirty = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [np.nan, 2, 3, np.nan, 5],
    'C': ['a', 'b', 'c', 'd', 'e']
})

print("Before fillna:")
print(df_dirty)

# Fill missing values
cleaned = HybridFrame.from_pandas(df_dirty).fillna(0)
print("\nAfter fillna:")
cleaned.to_pandas()

## 6. Feature Engineering

In [ ]:
# Add new columns
featured = (
    HybridFrame.from_pandas(df.head(100))
    .assign(
        salary_k=lambda d: d['salary'] / 1000,
        age_group=lambda d: pd.cut(d['age'], bins=[0, 25, 35, 50, 100], 
                                   labels=['Young', 'Mid', 'Senior', 'Expert'])
    )
)

featured.to_pandas().head()

## 7. SQL Passthrough

In [ ]:
# Raw SQL query
sql_result = (
    HybridFrame.from_pandas(df)
    .sql("""
        SELECT dept, 
               COUNT(*) as employee_count,
               ROUND(AVG(salary), 2) as avg_salary,
               ROUND(AVG(age), 1) as avg_age
        FROM self
        WHERE age > 25
        GROUP BY dept
        ORDER BY avg_salary DESC
    """)
)

sql_result.to_pandas()

## 8. ML-Ready Export

In [ ]:
# Prepare for ML
ml_data = (
    HybridFrame.from_pandas(df.head(1000))
    .filter("age > 25")
    .select(['age', 'salary'])
)

X, y = ml_data.to_ml_ready('salary')
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX sample:")
X.head()

## 9. Performance Comparison

In [ ]:
import time

# Benchmark: Filter + Sort + Head
def benchmark_pandas():
    return df[df['age'] > 30].sort_values('salary', ascending=False).head(10)

def benchmark_hybrid():
    return (
        HybridFrame.from_pandas(df)
        .filter('age > 30')
        .sort_values('salary', ascending=False)
        .head(10)
        .to_pandas()
    )

# Warm up
_ = benchmark_pandas()
_ = benchmark_hybrid()

# Benchmark
n_runs = 10

start = time.time()
for _ in range(n_runs):
    _ = benchmark_pandas()
pandas_time = (time.time() - start) / n_runs

start = time.time()
for _ in range(n_runs):
    _ = benchmark_hybrid()
hybrid_time = (time.time() - start) / n_runs

print(f"Pandas: {pandas_time*1000:.2f}ms")
print(f"HybridFrame: {hybrid_time*1000:.2f}ms")
print(f"Speedup: {pandas_time/hybrid_time:.2f}x")

## Summary

HybridFrame provides:
- **Zero-copy transitions** between DuckDB and Pandas
- **Lazy evaluation** for complex queries
- **Automatic engine selection** based on operation type
- **ML-ready export** with `to_ml_ready()`
- **SQL passthrough** for complex analytics
- **Memory-efficient streaming** with `fetch_chunked()`